# Bias of VIF-Laplace-Approximation

## Packages

In [ ]:
import torch
import gpytorch
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import os
import gpboost as gpb
import requests
import pandas as pd
import time

In [ ]:
# Flag for toy example
toy = True

## Data

In [ ]:
def simulate_gp_response(
    likelihood_type="gaussian", 
    sample_size=1000, 
    nugget=0.1, 
    marginal_variance=1.0, 
    custom_lengthscales=None, 
    seed=1,
    shape = 0.5
):
    """
    Simulate response and input variables from a Gaussian Process.

    Parameters:
    - likelihood_type: str, "gaussian" or "bernoulli-logit" for the likelihood.
    - sample_size: int, number of samples to generate.
    - nugget: float, noise variance (nugget).
    - marginal_variance: float, marginal variance (outputscale).
    - custom_lengthscale: list or tensor, custom length scales for each input dimension.
    - seed: int, random seed for reproducibility.

    Returns:
    - X: torch.Tensor, input variables.
    - sampled_field: torch.Tensor, simulated response.
    """
    
    torch.manual_seed(seed)
    if likelihood_type == "matern":
        class ExactGPModel(gpytorch.models.ExactGP):
            def __init__(self, train_x, train_y, likelihood):
                super().__init__(train_x, train_y, likelihood)
                self.mean_module = gpytorch.means.ConstantMean()
                self.covar_module = gpytorch.kernels.ScaleKernel(
                    gpytorch.kernels.MaternKernel(nu = shape,ard_num_dims=train_x.shape[1])  # Enable ARD
                )

            def forward(self, x):
                mean_x = self.mean_module(x)
                covar_x = self.covar_module(x)
                return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)
    else:
        class ExactGPModel(gpytorch.models.ExactGP):
            def __init__(self, train_x, train_y, likelihood):
                super().__init__(train_x, train_y, likelihood)
                self.mean_module = gpytorch.means.ConstantMean()
                self.covar_module = gpytorch.kernels.ScaleKernel(
                    gpytorch.kernels.RBFKernel(ard_num_dims=train_x.shape[1])  # Enable ARD
                )

            def forward(self, x):
                mean_x = self.mean_module(x)
                covar_x = self.covar_module(x)
                return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

    # Step 2: Dummy Training Data
    train_x = torch.rand(100, custom_lengthscales.shape[1])  # Small dummy train set
    train_y = torch.zeros(100)    # Dummy target values

    likelihood = gpytorch.likelihoods.GaussianLikelihood()
        
    model = ExactGPModel(train_x, train_y, likelihood)

    # Step 3: Set Custom Length Scales
    model.covar_module.base_kernel.lengthscale = custom_lengthscales

    # Adjust Marginal Variance (Outputscale)
    model.covar_module.outputscale = torch.tensor(marginal_variance)

    # Step 4: Switch to Evaluation Mode
    model.eval()
    likelihood.eval()

    # Step 5: Generate a Large Input Dataset
    n_points = sample_size  # Large dataset
    input_dim = custom_lengthscales.shape[1]
    X = torch.rand(n_points, input_dim)

    # Step 6: Sample from the GP Prior Without Constructing Full Covariance
    with torch.no_grad():
        latent_values = model(X).sample()

        if likelihood_type != "bernoulli-logit":
            sampled_field = latent_values + torch.randn_like(latent_values) * np.sqrt(nugget)
            probs = None  # No probabilities for Gaussian
        else:
            # Convert latent values to probabilities using the sigmoid function
            probs = torch.sigmoid(latent_values)
            # Simulate binary responses based on probabilities
            sampled_field = torch.bernoulli(probs)
        
    return X, sampled_field, latent_values


## Experiment

In [ ]:
# Different sample size
vector_n = [500,1000,2000,5000,7000,10000]
if toy:
    vector_n = [5000,7000]
# Number of reps 
num_reps = 100
if toy:
    num_reps = 10
# ranges
ranges = torch.tensor([(0.15, 0.3,0.45,0.6,0.75)])
# Zero matrix
matrix = np.zeros((num_reps, len(vector_n)))
# Nested loop to iterate over both vectors
for i, val1 in enumerate(vector_n):
    for j in range(num_reps):
        print(i)
        print(j)
        X, y, b = simulate_gp_response("bernoulli-logit",val1, 0, 1, ranges,j*10,1.5)
        data = pd.DataFrame(X.numpy(), columns=[f"x{i+1}" for i in range(X.shape[1])])
        data['y'] = y.numpy()
        # Select the first 5 columns for X
        X = data.iloc[:, :5]  # First 5 columns

        # Select the last column for y
        y = data.iloc[:, -1]  # Last column

        # Convert to numpy:
        X_np = X.to_numpy()
        y_np = y.to_numpy()
        print("Start Training")
        model_vif_eucl = gpb.GPModel(gp_coords=X_np, cov_function="gaussian_ard", num_neighbors = 30,
                                 likelihood="bernoulli_logit",num_ind_points = 200,ind_points_selection = "kmeans++",
                                 matrix_inversion_method = "iterative", gp_approx="full_scale_vecchia_correlation_based")
        model_vif_eucl.set_optim_params(params={"cg_preconditioner_type": "predictive_process_plus_diagonal",
                                            "fitc_piv_chol_preconditioner_rank": 200})
        
        model_vif_eucl.fit(y = y_np)
        matrix[j, i] = model_vif_eucl.get_cov_pars().iloc[0,0]
        
            